# Solutions — NYC Taxi Advanced Analytics (Hard 01)

In [ ]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window

trips = spark.table("samples.nyctaxi.trips")


## Solution 1 — Rush Hour Fare Premium

In [ ]:
result_1 = (
    trips
    .withColumn("hour", F.hour("tpep_pickup_datetime"))
    .withColumn("dow",  F.dayofweek("tpep_pickup_datetime"))   # 1=Sun … 7=Sat; 2-6 = Mon-Fri
    .withColumn("time_category",
        F.when(
            (F.col("dow").between(2, 6)) &
            ((F.col("hour").between(7, 8)) | (F.col("hour").between(17, 18))),
            "rush_hour"
        ).otherwise("off_peak")
    )
    .groupBy("time_category")
    .agg(
        F.count("*").alias("trip_count"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
        F.round(F.avg("trip_distance"), 2).alias("avg_distance"),
        F.round(F.sum("fare_amount"), 2).alias("total_revenue"),
    )
    .orderBy("time_category")
)


## Solution 2 — Trip Duration Anomaly Detection

In [ ]:
from pyspark.sql import functions as F

with_duration = trips.withColumn(
    "duration_minutes",
    F.round((F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60, 2)
)

stats = with_duration.agg(
    F.mean("duration_minutes").alias("mean_dur"),
    F.stddev("duration_minutes").alias("std_dur"),
).collect()[0]

mean_dur, std_dur = stats["mean_dur"], stats["std_dur"]

result_2 = (
    with_duration
    .withColumn("is_anomaly",
        (F.col("duration_minutes") > mean_dur + 2 * std_dur) |
        (F.col("duration_minutes") < mean_dur - 2 * std_dur)
    )
    .select("tpep_pickup_datetime", "trip_distance", "fare_amount", "duration_minutes", "is_anomaly")
)

## Solution 3 — Fare Efficiency Score

In [ ]:
w = Window.orderBy(F.col("avg_fare_per_mile").desc())
result_3 = (
    trips
    .filter(F.col("trip_distance") > 0)
    .groupBy("pickup_zip")
    .agg(
        F.round(F.avg(F.col("fare_amount") / F.col("trip_distance")), 4).alias("avg_fare_per_mile"),
        F.count("*").alias("trip_count"),
    )
    .withColumn("efficiency_rank", F.rank().over(w))
    .orderBy("efficiency_rank")
)


## Solution 4 — Busiest Pickup-Dropoff Corridors

In [ ]:
total_trips = trips.count()

result_4 = (
    trips
    .groupBy("pickup_zip", "dropoff_zip")
    .agg(
        F.count("*").alias("trip_count"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
        F.round(F.avg("trip_distance"), 2).alias("avg_distance"),
    )
    .withColumn("pct_of_total_trips", F.round(F.col("trip_count") / total_trips * 100, 4))
    .orderBy(F.col("trip_count").desc())
    .limit(20)
)

## Solution 5 — Hourly Trip Patterns by Day of Week

In [ ]:
w = Window.partitionBy("day_of_week").orderBy(F.col("trip_count").desc())

result_5 = (
    trips
    .withColumn("day_of_week", F.dayofweek("tpep_pickup_datetime"))
    .withColumn("hour_of_day",  F.hour("tpep_pickup_datetime"))
    .groupBy("day_of_week", "hour_of_day")
    .agg(F.count("*").alias("trip_count"))
    .withColumn("rank_within_day", F.rank().over(w))
    .orderBy("day_of_week", "rank_within_day")
)


## Solution 6 — Revenue Percentiles by Pickup Zip

In [ ]:
w = Window.orderBy(F.col("trip_count").desc())

result_6 = (
    trips
    .groupBy("pickup_zip")
    .agg(
        F.count("*").alias("trip_count"),
        F.round(F.percentile_approx("fare_amount", 0.25), 2).alias("p25"),
        F.round(F.percentile_approx("fare_amount", 0.50), 2).alias("p50"),
        F.round(F.percentile_approx("fare_amount", 0.75), 2).alias("p75"),
        F.round(F.percentile_approx("fare_amount", 0.95), 2).alias("p95"),
    )
    .orderBy(F.col("trip_count").desc())
    .limit(10)
)


## Solution 7 — Distance Bucket Analysis

In [ ]:
result_7 = (
    trips
    .filter(F.col("trip_distance") > 0)
    .withColumn("distance_bucket",
        F.when(F.col("trip_distance") < 1,  "<1mi")
         .when(F.col("trip_distance") < 3,  "1-3mi")
         .when(F.col("trip_distance") < 5,  "3-5mi")
         .when(F.col("trip_distance") < 10, "5-10mi")
         .otherwise(">10mi")
    )
    .withColumn("duration_minutes",
        (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60
    )
    .groupBy("distance_bucket")
    .agg(
        F.count("*").alias("trip_count"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
        F.round(F.avg("duration_minutes"), 2).alias("avg_duration_mins"),
        F.round(F.sum("fare_amount"), 2).alias("total_revenue"),
    )
    .orderBy("distance_bucket")
)

## Solution 8 — Rolling 7-Day Revenue per Pickup Zip

In [ ]:
w = Window.partitionBy("pickup_zip").orderBy("pickup_date").rowsBetween(-6, 0)

result_8 = (
    trips
    .withColumn("pickup_date", F.to_date("tpep_pickup_datetime"))
    .groupBy("pickup_date", "pickup_zip")
    .agg(F.round(F.sum("fare_amount"), 2).alias("daily_revenue"))
    .withColumn("rolling_7day_avg", F.round(F.avg("daily_revenue").over(w), 2))
    .orderBy("pickup_zip", "pickup_date")
)
